In [1]:
%reset -f 
import numpy as np
import matplotlib.pyplot as plt
import pooch

from batread import Dataset

from batcamp import OctreeInterpolator


/Users/dagfev/miniconda3/envs/batcamp/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 


In [ ]:
sc_path = pooch.retrieve(
    url="https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz",
    known_hash="c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136",
    progressbar=False,
    processor=pooch.Untar(members=["run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt"]),
)
if isinstance(sc_path, (list, tuple)):
    sc_path = sc_path[0]
print(sc_path)


In [ ]:
ih_path = pooch.retrieve(
    url="https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz",
    known_hash="c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136",
    progressbar=False,
    processor=pooch.Untar(members=["run-Sun-G2211/IH/IO2/3d__var_4_n00005000.plt"]),
)
if isinstance(ih_path, (list, tuple)):
    ih_path = ih_path[0]
print(ih_path)


In [ ]:
ds_sc = Dataset.from_file(sc_path)
ds_ih = Dataset.from_file(ih_path)
print(f"SC:
{ds_sc}
")
print(f"IH:
{ds_ih}
")


In [ ]:
interp_sc = OctreeInterpolator(ds_sc, ["Rho [g/cm^3]"])
interp_ih = OctreeInterpolator(ds_ih, ["Rho [g/cm^3]"])
print(f"SC:
{interp_sc}
")
print(f"IH:
{interp_ih}
")


In [ ]:
X, Y = np.meshgrid(np.linspace(-60, 60, 512), np.linspace(-60, 60, 512))
Z = np.zeros_like(X)
R = np.sqrt(X**2 + Y**2 + Z**2)

merge_radius = 50
rho = np.where(R <= merge_radius, interp_sc(X, Y, Z), interp_ih(X, Y, Z))

pcm = plt.pcolormesh(X, Y, rho, norm="log")
plt.colorbar(pcm)
plt.title("Merged SC/IH")
plt.show()


In [ ]:
R = np.geomspace(1, 150, 1024)
AZ = np.linspace(0, 2 * np.pi, 512)
R, AZ = np.meshgrid(R, AZ)
X = R * np.cos(AZ)
Y = R * np.sin(AZ)
Z = np.zeros_like(X)

rho = np.where(R <= merge_radius, interp_sc(X, Y, Z), interp_ih(X, Y, Z))

fig, ax = plt.subplots()
pcm = ax.pcolormesh(X, Y, rho, norm="log")
plt.colorbar(pcm)
ax.set_title("Merged SC/IH (spherical grid)")
ax.set_aspect("equal")
plt.show()
